<a href="https://colab.research.google.com/github/harigaran27official-beep/HARIGARAN/blob/main/IBM_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.ensemble import IsolationForest
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings

# Configuration and Styling
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

def generate_transaction_data():
    """Generates a synthetic dataset of normal and unusual transactions."""
    np.random.seed(42)

    # Normal Transactions
    normal_tx = pd.DataFrame({
        'Transaction_Amount': np.random.normal(loc=50.0, scale=20.0, size=1000),
        'Time_Hour': np.random.randint(6, 23, size=1000),
        'Distance_From_Home': np.random.normal(loc=8.0, scale=5.0, size=1000)
    })

    # Fraudulent Transactions
    fraud_tx = pd.DataFrame({
        'Transaction_Amount': np.random.normal(loc=4000.0, scale=800.0, size=25),
        'Time_Hour': np.random.choice([0, 1, 2, 3, 4], size=25),
        'Distance_From_Home': np.random.normal(loc=800.0, scale=200.0, size=25)
    })

    df = pd.concat([normal_tx, fraud_tx], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    df['Transaction_Amount'] = df['Transaction_Amount'].abs()
    df['Distance_From_Home'] = df['Distance_From_Home'].abs()
    return df

# Initialize Data and Model
data = generate_transaction_data()
features = ['Transaction_Amount', 'Time_Hour', 'Distance_From_Home']
model = IsolationForest(contamination=0.025, random_state=42)
data['Anomaly_Score'] = model.fit_predict(data[features])
data['Status'] = data['Anomaly_Score'].apply(lambda x: 'Suspicious' if x == -1 else 'Normal')

# --- ADVANCED VISUALIZATION ---
def plot_3d_analysis():
    fig = px.scatter_3d(data, x='Transaction_Amount', y='Distance_From_Home', z='Time_Hour',
                        color='Status', color_discrete_map={'Normal': '#00d2ff', 'Suspicious': '#ff4b2b'},
                        title="3D Fraud Pattern Analysis", opacity=0.7,
                        template='plotly_dark')
    fig.update_layout(margin=dict(l=0, r=0, b=0, t=40))
    fig.show()

# --- COOLER INTERACTIVE UI ---
output = widgets.Output()

# Styled inputs
amount_input = widgets.FloatText(value=50.0, description='Amount ($):', style={'description_width': 'initial'})
hour_input = widgets.IntSlider(value=12, min=0, max=23, step=1, description='Time (24h):', style={'description_width': 'initial'})
distance_input = widgets.FloatText(value=5.0, description='Dist. (mi):', style={'description_width': 'initial'})
check_button = widgets.Button(
    description="RUN SECURITY CHECK",
    button_style='danger',
    layout=widgets.Layout(width='95%', height='40px', margin='20px 0px')
)

def on_button_clicked(b):
    with output:
        clear_output()
        user_tx = pd.DataFrame([[amount_input.value, hour_input.value, distance_input.value]], columns=features)
        prediction = model.predict(user_tx)

        if prediction[0] == -1:
            display(HTML("""
                <div style="background-color: #ff4b2b; padding: 20px; border-radius: 10px; border: 2px solid #a83232; color: white; text-align: center;">
                    <h2 style="margin: 0;">🚨 FRAUD DETECTED 🚨</h2>
                    <p style="font-size: 1.2em;">This transaction matches high-risk anomaly patterns!</p>
                    <div style="font-weight: bold; font-size: 1.5em; border-top: 1px solid white; margin-top: 10px; padding-top: 10px;">STATUS: BLOCKED</div>
                </div>
            """))
        else:
            display(HTML("""
                <div style="background-color: #28a745; padding: 20px; border-radius: 10px; border: 2px solid #1e7e34; color: white; text-align: center;">
                    <h2 style="margin: 0;">✅ TRANSACTION SECURE</h2>
                    <p style="font-size: 1.2em;">Activity verified against historical norms.</p>
                    <div style="font-weight: bold; font-size: 1.5em; border-top: 1px solid white; margin-top: 10px; padding-top: 10px;">STATUS: APPROVED</div>
                </div>
            """))

check_button.on_click(on_button_clicked)

# --- LAYOUT ---
display(HTML("<h1 style='color: #ff4b2b; text-align: center; font-family: sans-serif;'>CYBER-GUARD FRAUD ENGINE</h1>"))
plot_3d_analysis()

ui_container = widgets.VBox([
    widgets.HTML("<h3 style='margin-left: 10px; color: #555;'>Transaction Manual Entry</h3>"),
    widgets.HBox([amount_input, hour_input, distance_input]),
    check_button
], layout=widgets.Layout(border='1px solid #ddd', padding='20px', border_radius='10px', margin='20px 0px'))

display(ui_container, output)

# Final stats badge
display(HTML(f"""
    <div style='background: #333; color: white; padding: 10px; border-radius: 5px; width: fit-content;'>
        Engine Stats: {len(data[data['Status']=='Suspicious'])} Anomaly Vectors Loaded | Total Pool: {len(data)}
    </div>
"""))

Output()